# The Draft Lottery

**Complete | 200pts**
*Instructions | Submissions | Resources*

Phase estimation is usually introduced with a promise attached: you are handed an eigenvector of the unitary operator you are using, and the algorithm hands back its eigenphase. That promise is almost never kept in practice. One rarely knows the eigenvectors of the operator you care about. If you did, you would already have its spectrum. What actually goes into the target register is some state that happens to overlap several eigenvectors at once.

The good news is that the algorithm still works, and it even does something more useful.

## Leveraging linearity

Let $U$ have eigenvectors $\vert{}u_k\rangle$ with eigenphases $\theta_k$, so that $U\vert{}u_k\rangle = e^{2\pi i \theta_k}\vert{}u_k\rangle$. Phase estimation with $m$ estimation wires acts as

$$\vert{}u_k\rangle\vert{}0\rangle^{\otimes m} \longmapsto \vert{}u_k\rangle\vert{}\tilde{\theta}_k\rangle,$$

where $\vert{}\tilde{\theta}_k\rangle$ is the estimation register holding the $m$-bit approximation of $\theta_k$. The circuit is linear, so feeding it a superposition feeds every branch through at once. Writing the target state as $\vert{}\psi\rangle = \sum_k c_k \vert{}u_k\rangle$:

$$\vert{}\psi\rangle\vert{}0\rangle^{\otimes m} \longmapsto \sum_k c_k \vert{}u_k\rangle\vert{}\tilde{\theta}_k\rangle.$$

Now measure only the estimation register. The target register is left alone, so the branches cannot interfere, and the probability of reading the bit string $\tilde{\theta}_k$ is given by

$$\Pr[\tilde{\theta}_k] = \vert{}c_k\vert{}^2.$$

One run of the circuit therefore samples an eigenphase, with each eigenphase appearing in proportion to how much of it your input state contained. The estimation register's probability distribution is a picture of the spectrum, weighted by overlap. This is the mechanism behind quantum eigensolvers: prepare a state you hope resembles your state of interest, run phase estimation, and the eigenvalue/eigenphase of your state of interest shows up with probability equal to the squared overlap you managed to achieve.

Two consequences follow, and this challenge is about both.

**Reading the spectrum.** Bin $k$ of the estimation register corresponds to the phase $k/2^m$. If you collect every bin with non-negligible probability, you have the eigenphases present, along with their weights. If two different eigenvectors share the same eigenphase, they land in the same bin and their weights add. Keep in mind the register measures phases, not eigenvectors, and cannot tell them apart.

**Paying for the faint ones.** An eigenphase with weight $w$ appears in a single run with probability $w$. That means that a weight of one percent is not a small inconvenience, it is a factor of a hundred extra shots in runtime. Specifically, to see such an eigenphase at least once with probability $99\%$ you need $r$ independent repetitions, where

$$(1-w)^r \leq 0.01 \implies r \geq \frac{\ln 0.01}{\ln(1-w)},$$

rounded up. This is why overlap, not just circuit depth, is a critical characteristic in eigenvalue estimation and why a poor initial guess can be more expensive than a deeper circuit.

## Challenge code

Write

`read_spectrum(phases, amplitudes, m)`

The Challenge Template gives you `unitary_matrix(phases)`, which builds a diagonal unitary whose eigenvalue on basis state $\vert{}k\rangle$ is $e^{2\pi i \theta_k}$. Because it is diagonal, its eigenvectors are the computational-basis states, which makes the weights easy to reason about but does not change anything about the algorithm.

Build a device with the target wires first and the $m$ estimation wires after them, prepare amplitudes on the target register, run phase estimation, and measure the estimation register.

### Inputs

Your function is called with three arguments:

* `phases` -- `list[float]`. One eigenphase in $[0,1)$ per computational-basis state of the target register. `len(phases)` is a power of two, so `len(phases) == 2` means one target wire and `len(phases) == 4` means two.
* `amplitudes` -- `list[float]`. Real, normalised amplitudes for the target register, with `len(amplitudes) == len(phases)`. Entry `amplitudes[k]` sits on basis state $\vert{}k\rangle$, so `amplitudes[k] ** 2` is the weight of eigenphase `phases[k]`. Pass this list straight to a state preparation.
* `m` -- `int`. The number of estimation wires.

So a call looks like this:

```python
read_spectrum([0.1875, 0.6875], [0.8, 0.6], 4)

```

which is one target wire holding $0.8\vert{}0\rangle + 0.6\vert{}1\rangle$, read with four estimation wires.

Every test case uses eigenphases that are exact multiples of $1/2^m$, so each one lands in a single bin and the peaks are sharp. You do not have to deal with spectral leakage here.

### Outputs

Return a plain Python tuple of three items, in this order:

`return eigenphases, weights, repetitions`

* `eigenphases` -- `list[float]`. The distinct phases present, in ascending order. Keep a bin if its probability exceeds `1e-9`.
* `weights` -- `list[float]`. The total probability of each phase, in the same order, so that `weights[i]` belongs to `eigenphases[i]`.
* `repetitions` -- `int`. The number of independent runs needed to observe the faintest phase at least once with probability at least $99\%$.

### Test cases

These two cases are public, and both are shown in full. The remaining cases are private, and run only when you submit.

```python
read_spectrum([0.1875, 0.6875], [0.8, 0.6], 4)
# -> ([0.1875, 0.6875], [0.6399999999999999, 0.3599999999999996], 11)

read_spectrum([0.125, 0.375, 0.625, 0.875], [0.7, 0.5, 0.5, 0.1], 3)
# -> ([0.125, 0.375, 0.625, 0.875], [0.4899999999999996, 0.2499999999999999, 0.25, 0.009999999999999993], 459)

```

The weights carry the usual floating-point noise -- `0.6399999999999999` is $0.64$, and `0.009999999999999993` is one percent -- so do not try to reproduce those digits. They are compared to a tolerance.

In the first case one target wire carries $0.8\vert{}0\rangle + 0.6\vert{}1\rangle$, so the two eigenphases arrive with weights $0.64$ and $0.36$; the faintest needs $11$ repetitions. The second case has four eigenphases, one of which holds only one percent of the state -- and that single faint phase drives the cost to $459$ repetitions, more than forty times the first case, on a shallower circuit.

Eigenphases are checked to an absolute tolerance of $10^{-9}$ and weights to $10^{-6}$. The repetition count must match exactly.

In [ ]:
import json
import pennylane as qp
import pennylane.numpy as np

# The unitary whose spectrum you have to read. You do not need to change this.
def unitary_matrix(phases):
    """Builds a diagonal unitary with the given eigenphases.

    The eigenvalue on computational-basis state |k> is exp(2j * pi * phases[k]),
    so the eigenvectors are the computational-basis states themselves.

    Args:
        phases (list[float]): eigenphases in [0, 1), one per basis state. The length
            is a power of two.

    Returns:
        numpy array: the diagonal unitary matrix.
    """
    return np.diag(np.array([np.exp(2j * np.pi * phase) for phase in phases]))



def read_spectrum(phases, amplitudes, m):
    """Reads every eigenphase present in a superposition, together with its weight,
    and prices the repetitions needed to see the faintest one.

    Args:
        phases (list[float]): the eigenphases of the unitary, passed to unitary_matrix.
        amplitudes (list[float]): a normalised real state on the target wires.
        m (int): the number of estimation wires.

    Returns:
        tuple[list[float], list[float], int]:
            - the eigenphases present, ascending;
            - the weight of each, in the same order;
            - the number of repetitions needed to see the faintest one.
    """


# Put your code here.
    n = int(round(np.log2(len(phases))))
    dev = qp.device("default.qubit", wires=n + m)

    @qp.qnode(dev)
    def circuit():
        qp.StatePrep(np.array(amplitudes), wires=range(n))
        qp.QuantumPhaseEstimation(
            unitary_matrix(phases),
            target_wires=range(n),
            estimation_wires=range(n, n + m),
        )
        return qp.probs(wires=range(n, n + m))

    probs = circuit()

    eigenphases = []
    weights = []
    for k, p in enumerate(probs):
        p = float(p)
        if p > 1e-9:
            eigenphases.append(k / 2 ** m)
            weights.append(p)

    w_min = min(weights)
    if w_min >= 1.0:
        repetitions = 1
    else:
        repetitions = int(np.ceil(np.log(0.01) / np.log(1 - w_min)))

    return eigenphases, weights, repetitions
    eigenphases = None
    weights = None
    repetitions = None
    return eigenphases, weights, repetitions

# These functions are responsible for testing the solution.
#
def run(test_case_input: str) -> str:
    loads, dumps = json.loads, json.dumps
    make_device, script = qp.device, qp.tape.QuantumScript
    phases, amplitudes, m = loads(test_case_input)
    log = []

    def audited_device(*args, **kwargs):
        """A default.qubit that reports what it ran and what it returned."""
        device = make_device(*args, **kwargs)
        real_execute = device.execute

        def execute(circuits, *inner_args, **inner_kwargs):
            results = real_execute(circuits, *inner_args, **inner_kwargs)
            single = isinstance(circuits, script)
            batch = [circuits] if single else list(circuits)
            outputs = [results] if single else list(results)
            for tape, output in zip(batch, outputs):
                if len(tape.measurements) != 1:
                    continue
                measurement = tape.measurements[0]
                log.append((type(measurement).__name__, tape.shots.total_shots,
                            tuple(measurement.wires),
                            [float(value) for value in np.asarray(output).ravel()]))
            return results

        device.execute = execute
        return device

    qp.device = audited_device
    try:
        returned = read_spectrum(phases, amplitudes, m)
    finally:
        qp.device = make_device

    try:
        eigenphases, weights, repetitions = returned
    except (TypeError, ValueError):
        raise AssertionError(
            "Return exactly three values: the eigenphases, their weights, and the "
            "number of repetitions.")
    eigenphases = [float(phase) for phase in eigenphases]
    weights = [float(weight) for weight in weights]
    repetitions = int(repetitions)
    if len(eigenphases) != len(weights):
        raise AssertionError(
            "You returned " + str(len(eigenphases)) + " eigenphase(s) but "
            + str(len(weights)) + " weight(s). One weight per eigenphase.")
    if sorted(eigenphases) != eigenphases:
        raise AssertionError("Return the eigenphases in ascending order.")

    for kind, _, _, _ in log:
        if kind != "ProbabilityMP":
            raise AssertionError(
                "A circuit returned " + kind + ". The estimation register has to be "
                "read with qp.probs so that every eigenphase present shows up.")

    # Rebuild the answer from what the device actually returned. Any distribution the
    # solution measured over exactly m wires is admissible; at least one of them has
    # to reproduce the eigenphases and weights being reported.
    reproduced = False
    for kind, shots, wires, distribution in log:
        if shots is not None or len(wires) != m or len(distribution) != 2 ** m:
            continue
        peaks = {}
        for k, weight in enumerate(distribution):
            if weight > 1e-9:
                phase = k / 2 ** m
                peaks[phase] = peaks.get(phase, 0.0) + weight
        found = sorted(peaks)
        if (len(found) == len(eigenphases)
                and all(abs(a - b) < 1e-12 for a, b in zip(found, eigenphases))
                and all(abs(peaks[phase] - weight) < 1e-6
                        for phase, weight in zip(found, weights))):
            reproduced = True
            break
    if not reproduced:
        raise AssertionError(
            "The eigenphases and weights you returned do not match any distribution "
            "your circuit measured. Run phase estimation with " + str(m)
            + " estimation wires on the given superposition, return "
            "qp.probs of the estimation wires, and read the peaks off that -- "
            + str(len(log)) + " execution(s) were recorded.")
    faintest = min(weights)
    if repetitions != int(np.ceil(np.log(0.01) / np.log(1.0 - faintest))):
        raise AssertionError(
            "Wrong repetition count -- solve (1 - w)**r <= 0.01 for the faintest "
            "weight w and round up.")
    return dumps([eigenphases, weights, repetitions])


def check(solution_output: str, expected_output: str) -> None:
    got = json.loads(solution_output)
    exp = json.loads(expected_output)
    assert len(got[0]) == len(exp[0]), \
        "Wrong number of eigenphases -- keep every bin whose probability exceeds " \
        "1e-9, and remember that two eigenvectors sharing an eigenphase produce " \
        "one peak, not two."
    assert np.allclose(got[0], exp[0], rtol=0.0, atol=1e-9), \
        "Wrong eigenphases -- bin k of the estimation register corresponds to the " \
        "phase k / 2**m."
    assert np.allclose(got[1], exp[1], rtol=0.0, atol=1e-6), \
        "Wrong weights -- the probability of a peak is the squared overlap of the " \
        "input state with the corresponding eigenspace."
    assert got[2] == exp[2], \
        "Wrong repetition count -- solve (1 - w)**r <= 0.01 for the faintest " \
        "weight w and round up."


# These are the public test cases
test_cases = [
    ('[[0.1875, 0.6875], [0.8, 0.6], 4]', '[[0.1875, 0.6875], [0.6399999999999999, 0.3599999999999996], 11]'),
    ('[[0.125, 0.375, 0.625, 0.875], [0.7, 0.5, 0.5, 0.1], 3]', '[[0.125, 0.375, 0.625, 0.875], [0.4899999999999996, 0.2499999999999999, 0.25, 0.009999999999999993], 459]')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")